# A) Les imports

In [1]:
# ÉTAPE 1 : Préparation de l'environnement
import pandas as pd
import os

# 1. Définir le chemin du dossier contenant les fichiers
chemin_dossier = r"C:\Users\hp\Desktop\works\donnees\pourvisuel"
print(f"Dossier source : {chemin_dossier}")

# 2. Vérifier que le dossier existe
if not os.path.isdir(chemin_dossier):
    print("❌ ERREUR : Le dossier n'existe pas. Vérifiez le chemin.")
else:
    print("✅ Dossier vérifié")

# 3. Liste des mois à charger (dans l'ordre chronologique)
mois_a_charger = ['Aug2025', 'Sep2025', 'Oct2025', 'Nov2025', 'Dec2025']
print(f"Mois à charger : {mois_a_charger}")

# 4. Préparer le dictionnaire qui stockera les DataFrames
dataframes_par_mois = {}

print("\n" + "="*50)
print("ÉTAPE 1 TERMINÉE : Environnement prêt.")
print("Prochaine étape : Chargement des fichiers.")

Dossier source : C:\Users\hp\Desktop\works\donnees\pourvisuel
✅ Dossier vérifié
Mois à charger : ['Aug2025', 'Sep2025', 'Oct2025', 'Nov2025', 'Dec2025']

ÉTAPE 1 TERMINÉE : Environnement prêt.
Prochaine étape : Chargement des fichiers.


In [2]:
# ÉTAPE 2 : Chargement des fichiers
print("="*50)
print("ÉTAPE 2 : CHARGEMENT DES FICHIERS")
print("="*50)

fichiers_manquants = []
fichiers_charges = 0

for mois in mois_a_charger:
    # 1. Construire le nom du fichier
    nom_fichier = f"visuel_3966_{mois}.csv"
    chemin_fichier = os.path.join(chemin_dossier, nom_fichier)
    
    # 2. Vérifier si le fichier existe
    if not os.path.exists(chemin_fichier):
        print(f"❌ Fichier non trouvé : {nom_fichier}")
        fichiers_manquants.append(nom_fichier)
        continue
    
    # 3. Charger le fichier CSV
    try:
        df = pd.read_csv(chemin_fichier)
        dataframes_par_mois[mois] = df
        fichiers_charges += 1
        
        # 4. Afficher les informations du fichier
        print(f"✅ {nom_fichier}")
        print(f"   → Lignes : {df.shape[0]:>6} | Colonnes : {df.shape[1]:>3}")
        print(f"   → Période : {df['timestamp'].min()} à {df['timestamp'].max()}")
        
    except Exception as e:
        print(f"❌ Erreur lors du chargement de {nom_fichier} : {e}")
        fichiers_manquants.append(nom_fichier)

# 5. Résumé du chargement
print("\n" + "="*50)
print("RÉSUMÉ DU CHARGEMENT :")
print(f"Fichiers chargés avec succès : {fichiers_charges}/{len(mois_a_charger)}")

if fichiers_charges == len(mois_a_charger):
    print("✅ TOUS LES FICHIERS ONT ÉTÉ CHARGÉS")
    print("Prochaine étape : Concaténation et tri.")
elif fichiers_charges > 0:
    print(f"⚠️  {len(fichiers_manquants)} fichier(s) manquant(s) : {fichiers_manquants}")
    print("Voulez-vous continuer avec les fichiers disponibles ?")
else:
    print("❌ AUCUN FICHIER N'A PU ÊTRE CHARGÉ. Arrêt du processus.")

ÉTAPE 2 : CHARGEMENT DES FICHIERS
✅ visuel_3966_Aug2025.csv
   → Lignes :   2214 | Colonnes :  12
   → Période : 2025-08-01 03:42:05.214502500+00:00 à 2025-08-31 18:51:12.835876500+00:00
✅ visuel_3966_Sep2025.csv
   → Lignes :   2503 | Colonnes :  12
   → Période : 2025-09-01 03:28:21.236928+00:00 à 2025-09-30 20:31:25.767538+00:00
✅ visuel_3966_Oct2025.csv
   → Lignes :   4550 | Colonnes :  12
   → Période : 2025-10-01 03:11:01.882746+00:00 à 2025-10-31 16:58:18.560406500+00:00
✅ visuel_3966_Nov2025.csv
   → Lignes :   4532 | Colonnes :  12
   → Période : 2025-11-01 03:26:35.054092+00:00 à 2025-11-30 23:47:49.664770+00:00
✅ visuel_3966_Dec2025.csv
   → Lignes :   7377 | Colonnes :  12
   → Période : 2025-12-01 00:04:37.210635+00:00 à 2025-12-31 23:56:33.319772+00:00

RÉSUMÉ DU CHARGEMENT :
Fichiers chargés avec succès : 5/5
✅ TOUS LES FICHIERS ONT ÉTÉ CHARGÉS
Prochaine étape : Concaténation et tri.


# B) Concatenation

In [3]:
# ÉTAPE 3 : Concaténation et tri
df_complet = pd.concat(dataframes_par_mois.values(), ignore_index=True)
df_complet = df_complet.sort_values('timestamp').reset_index(drop=True)

print(f"✅ Concaténation terminée : {len(df_complet)} lignes, {df_complet.shape[1]} colonnes")
print(f"📅 Période : {df_complet['timestamp'].iloc[0]} à {df_complet['timestamp'].iloc[-1]}")

✅ Concaténation terminée : 21176 lignes, 12 colonnes
📅 Période : 2025-08-01 03:42:05.214502500+00:00 à 2025-12-31 23:56:33.319772+00:00


# C) Re-echantillonage pour equilibre des frequences

In [5]:
# 1. Vérifier les colonnes non-numériques
print("Colonnes non-numériques :")
print(df_complet.select_dtypes(include=['object']).columns.tolist())

# 2. Convertir les colonnes numériques potentiellement en 'object'
colonnes_numeriques = ['P0', 'P1', 'P2', 'humidity', 'temperature']
for col in colonnes_numeriques:
    if col in df_complet.columns:
        df_complet[col] = pd.to_numeric(df_complet[col], errors='coerce')

# 3. Sélectionner uniquement les colonnes numériques pour le ré-échantillonnage
df_numerique = df_complet[colonnes_numeriques].copy()
df_resample = df_numerique.resample('1H').mean()
df_resample = df_resample.interpolate(method='time')
df_resample = df_resample.reset_index()

print(f"✅ Ré-échantillonnage terminé : {len(df_resample)} lignes")

Colonnes non-numériques :
['jour_semaine', 'periode']
✅ Ré-échantillonnage terminé : 3669 lignes


C:\Users\hp\AppData\Local\Temp\ipykernel_7340\671656405.py:13: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_resample = df_numerique.resample('1H').mean()


# D) Variables temporelles perdues

In [8]:
# ÉTAPE 5 : Lags et variables temporelles
# 1. Recréer les variables temporelles à partir du timestamp régulier
df_resample['heure'] = df_resample['timestamp'].dt.hour
df_resample['jour_semaine'] = df_resample['timestamp'].dt.day_name()
df_resample['periode'] = df_resample['jour_semaine'].apply(lambda x: 'weekend' if x in ['Saturday', 'Sunday'] else 'semaine')

# 2. Créer les lag features pour P2
lags = [1, 2, 3, 24, 168]  # 1h, 2h, 3h, 24h, 1 semaine
for lag in lags:
    df_resample[f'P2_lag_{lag}'] = df_resample['P2'].shift(lag)

# 3. Supprimer les lignes avec NaN des lags
df_clean = df_resample.dropna().reset_index(drop=True)

print(f"✅ Features créées : {len(df_clean)} lignes finales")
print(f"📊 Colonnes ajoutées : P2_lag_1, P2_lag_2, P2_lag_3, P2_lag_24, P2_lag_168")
print(f"🔄 Variables temporelles recréées : heure, jour_semaine, periode")

✅ Features créées : 3501 lignes finales
📊 Colonnes ajoutées : P2_lag_1, P2_lag_2, P2_lag_3, P2_lag_24, P2_lag_168
🔄 Variables temporelles recréées : heure, jour_semaine, periode


# E) Sauvegarde

In [9]:
# ÉTAPE 6 : Sauvegarde
chemin_final = r"C:\Users\hp\Desktop\works\donnees\modele_3966_Aug_Dec2025.csv"
df_clean.to_csv(chemin_final, index=False)

print(f"✅ Fichier sauvegardé : {chemin_final}")
print(f"📁 Taille : {len(df_clean)} lignes × {df_clean.shape[1]} colonnes")
print(f"📅 Période : {df_clean['timestamp'].iloc[0]} à {df_clean['timestamp'].iloc[-1]}")

# Vérification
taille_fichier = os.path.getsize(chemin_final) / (1024*1024)  # Taille en Mo
print(f"💾 Taille du fichier : {taille_fichier:.2f} Mo")

✅ Fichier sauvegardé : C:\Users\hp\Desktop\works\donnees\modele_3966_Aug_Dec2025.csv
📁 Taille : 3501 lignes × 14 colonnes
📅 Période : 2025-08-08 03:00:00+00:00 à 2025-12-31 23:00:00+00:00
💾 Taille du fichier : 0.70 Mo


In [11]:
df_clean.columns

Index(['timestamp', 'P0', 'P1', 'P2', 'humidity', 'temperature', 'heure',
       'jour_semaine', 'periode', 'P2_lag_1', 'P2_lag_2', 'P2_lag_3',
       'P2_lag_24', 'P2_lag_168'],
      dtype='object')

In [12]:
df_clean.head(5)

,timestamp,P0,P1,P2,humidity,temperature,heure,jour_semaine,periode,P2_lag_1,P2_lag_2,P2_lag_3,P2_lag_24,P2_lag_168
0,2025-08-08 03:00:00+00:00,22.111431,36.489514,30.930691,84.350000,15.250000,3,Friday,semaine,33.375576,35.820461,38.265346,13.900000,34.6650
1,2025-08-08 04:00:00+00:00,23.508573,40.748386,32.879269,72.721523,18.704283,4,Friday,semaine,30.930691,33.375576,35.820461,18.950000,31.8325
2,2025-08-08 05:00:00+00:00,24.905715,45.007257,34.827846,61.093046,22.158567,5,Friday,semaine,32.879269,30.930691,33.375576,24.000000,29.0000
3,2025-08-08 06:00:00+00:00,24.704000,50.080000,35.366000,51.743310,25.460561,6,Friday,semaine,34.827846,32.879269,30.930691,21.427846,24.0000
4,2025-08-08 07:00:00+00:00,40.081250,70.162500,54.506250,51.725000,25.150000,7,Friday,semaine,35.366000,34.827846,32.879269,24.236138,23.1000
